DSI Visualization 1 - Assignment 3
1. Enrollment in Public and Catholic Schools in Ontario - Graph Using Ministry of Education Data

In [32]:
import requests
import zipfile
import io
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [33]:

# Download the ZIP file
url = "https://data.ontario.ca/dataset/acc1ff32-3995-469d-9e94-adedf17f9c4e/resource/2a1c0f32-22b9-4bde-bc79-499b460282d5/download/en_2023-2024.zip"
response = requests.get(url)
response.raise_for_status()

# Extract ZIP contents
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    extract_path = "en_2023_2024_extracted"
    os.makedirs(extract_path, exist_ok=True)
    z.extractall(path=extract_path)

# List the extracted files
print("Extracted files:")
for name in os.listdir(extract_path):
    print(name)

Extracted files:
EN_2023-2024


In [34]:
# Locate the Excel file
folder_path = os.path.join(extract_path, "EN_2023-2024")

# List all files inside the extracted folder
print("Files inside EN_2023-2024:")
for filename in os.listdir(folder_path):
    print(filename)

Files inside EN_2023-2024:
14_Financial_Summary_2023-2024_EN.xlsx
9a_Teachers_Admin_ECE_Elem_2023-2024_EN.xlsx
8a_Teachers_Admin_FTE_2023-2024_EN.xlsx
5b. French_Language_Sch_Enrol_2023-2024_EN.xlsx
3-4_Enrolment_Grade_2023-2024_EN.xlsx
8b_Teachers_Admin_ECE_2023-2024_EN.xlsx
5. ElemSec_Enrol_2023-2024_EN.xlsx
2. Number_Of_Schools_2023-2024_EN_.xlsx
9b_Teachers_Admin_Sec_2023-2024_EN.xlsx
12_Number_of_Private_Schools_2023-2024_EN.xlsx
7. FSL_Enrolment_2023-2024_EN.xlsx


In [35]:
# Set the path to the extracted Excel file - Use enrollment data
file_path = os.path.join(folder_path, "3-4_Enrolment_Grade_2023-2024_EN.xlsx")

# Load Excel data
df = pd.read_excel(file_path)

In [36]:
# Filter only the total rows
totals_df = df[df["Grades"].isin(["Elementary - Total", "Secondary - Total"])].copy()

# Clean commas and convert to integers
totals_df["Enrolment - Public"] = totals_df["Enrolment - Public"].astype(str).str.replace(",", "").astype(int)
totals_df["Enrolment - Roman Catholic"] = totals_df["Enrolment - Roman Catholic"].astype(str).str.replace(",", "").astype(int)

# Prepare data for plotting
levels = ["Elementary", "Secondary"]
public = totals_df["Enrolment - Public"].tolist()
catholic = totals_df["Enrolment - Roman Catholic"].tolist()

bar_width = 0.35
x = np.arange(len(levels))  # [0, 1]

In [37]:
# Create Plotly bar chart
fig = go.Figure()

fig.add_trace(go.Bar(
    x=x - bar_width / 2,
    y=public,
    name="Public",
    marker_color="#4C72B0",
    hovertemplate=[
        f"Public<br>{(val / (public[i] + catholic[i]) * 100):.1f}% of total<extra></extra>"
        for i, val in enumerate(public)
    ]
))

fig.add_trace(go.Bar(
    x=x + bar_width / 2,
    y=catholic,
    name="Roman Catholic",
    marker_color="#55A868",
    hovertemplate=[
        f"Roman Catholic<br>{(val / (public[i] + catholic[i]) * 100):.1f}% of total<extra></extra>"
        for i, val in enumerate(catholic)
    ]
))

# Configure layout
fig.update_layout(
    title="2023–24 Ontario School Enrolment by Level and System",
    title_font_size=16,
    barmode='group',
    xaxis=dict(
        tickvals=x,
        ticktext=levels,
        title=''
    ),
    yaxis=dict(
        title="Number of Students",
        showgrid=False,
        tickformat=",",
        range=[0, max(max(public), max(catholic)) * 1.15]
    ),
    plot_bgcolor='white',
    bargap=0.2,
    legend_title_text='',
    margin=dict(t=80, l=60, r=40, b=60)
)

fig.show()
